# 🚀 CASSI GOAT 1.0 — SOTA Pipeline v2 (NaN-safe)

**Fixes from v1:**
1. ✅ SAM loss: cosine distance instead of `acos` (NaN in fp16)
2. ✅ Warmup: scheduler called BEFORE each epoch
3. ✅ All losses forced to float32
4. ✅ NaN-safe gradient updates (skip if NaN detected)
5. ✅ Updated `torch.amp` API (no deprecation warnings)
6. ✅ Gradient norm monitoring


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
import random
import math
import os
import gc

## ⚙️ Configuration

In [ ]:
class Config:
    DATA_DIR = Path("/kaggle/input/cassi-goat1-0-hackers/all_data")
    TRAIN_CODED_DIR = DATA_DIR / "train" / "coded_hsi"
    TRAIN_HS_DIR = DATA_DIR / "train" / "hs_cube"
    TEST_CODED_DIR = DATA_DIR / "test" / "coded_hsi"
    MASK_PATH = DATA_DIR / "mask_cube.pt"
    OUTPUT_DIR = Path("/kaggle/working")
    CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
    PRED_DIR = OUTPUT_DIR / "test_reconstructions"

    IN_CHANNELS = 58
    OUT_CHANNELS = 29
    BASE_DIM = 64
    NUM_HEADS = 4
    NUM_BLOCKS = [2, 2, 4, 2]
    
    EPOCHS = 80
    BATCH_SIZE = 8
    GRAD_ACCUM = 4
    LR = 2e-4               # lowered for stability
    MIN_LR = 1e-6
    WEIGHT_DECAY = 1e-4
    WARMUP_EPOCHS = 5
    
    L1_WEIGHT = 1.0
    SSIM_WEIGHT = 0.3
    SAM_WEIGHT = 0.1        # lowered — SAM loss is less stable
    FORWARD_WEIGHT = 0.05
    
    VAL_RATIO = 0.1
    NUM_WORKERS = 2
    SEED = 42
    USE_TTA = True

cfg = Config()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(cfg.SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 🔬 Load Mask + Physics Init

In [ ]:
mask_cube = torch.load(cfg.MASK_PATH, weights_only=True).float()
if mask_cube.shape[-1] == 29 and mask_cube.dim() == 3:
    mask_cube = mask_cube.permute(2, 0, 1)
print(f"Mask cube shape: {mask_cube.shape}")
print(f"Mask range: [{mask_cube.min():.1f}, {mask_cube.max():.1f}]")

def cassi_init(coded_measurement, mask):
    if coded_measurement.dim() == 2:
        coded_measurement = coded_measurement.unsqueeze(0)
    return coded_measurement * mask

## 📦 Dataset

In [ ]:
class CASSIDataset(Dataset):
    def __init__(self, coded_dir, hs_dir, mask, augment=False):
        self.coded_files = sorted(Path(coded_dir).glob("*.pt"))
        self.hs_files = sorted(Path(hs_dir).glob("*.pt"))
        self.mask = mask
        self.augment = augment
        assert len(self.coded_files) == len(self.hs_files)
    
    def __len__(self):
        return len(self.coded_files)
    
    def __getitem__(self, idx):
        coded = torch.load(self.coded_files[idx], weights_only=True).float()
        hs = torch.load(self.hs_files[idx], weights_only=True).float()
        
        if coded.dim() == 2:
            coded = coded.unsqueeze(0)
        if hs.dim() == 3 and hs.shape[-1] == 29:
            hs = hs.permute(2, 0, 1)
        
        init = cassi_init(coded, self.mask)
        model_input = torch.cat([init, self.mask], dim=0)
        hs = torch.clamp(hs, 0.0, 1.0)
        
        if self.augment:
            if random.random() < 0.5:
                model_input = torch.flip(model_input, [-1])
                hs = torch.flip(hs, [-1])
            if random.random() < 0.5:
                model_input = torch.flip(model_input, [-2])
                hs = torch.flip(hs, [-2])
            k = random.randint(0, 3)
            if k > 0:
                model_input = torch.rot90(model_input, k, [-2, -1])
                hs = torch.rot90(hs, k, [-2, -1])
        
        return model_input, hs, coded


class CASSITestDataset(Dataset):
    def __init__(self, coded_dir, mask):
        self.files = sorted(Path(coded_dir).glob("*.pt"))
        self.mask = mask
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        path = self.files[idx]
        coded = torch.load(path, weights_only=True).float()
        if coded.dim() == 2:
            coded = coded.unsqueeze(0)
        init = cassi_init(coded, self.mask)
        model_input = torch.cat([init, self.mask], dim=0)
        return model_input, coded, path.stem

## 🧠 Model Blocks (Restormer-inspired)

In [ ]:
class LayerNorm2d(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.GroupNorm(1, dim)
    def forward(self, x):
        return self.norm(x)

class SpectralChannelAttention(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, mid), nn.GELU(),
            nn.Linear(mid, channels), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.net(x).unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], 1)))

class MDTA(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        self.qkv = nn.Conv2d(dim, dim * 3, 1, bias=False)
        self.qkv_dw = nn.Conv2d(dim * 3, dim * 3, 3, padding=1, groups=dim * 3, bias=False)
        self.proj = nn.Conv2d(dim, dim, 1, bias=False)
    
    def forward(self, x):
        B, C, H, W = x.shape
        qkv = self.qkv_dw(self.qkv(x))
        q, k, v = qkv.chunk(3, dim=1)
        q = q.reshape(B, self.num_heads, -1, H * W)
        k = k.reshape(B, self.num_heads, -1, H * W)
        v = v.reshape(B, self.num_heads, -1, H * W)
        q, k = F.normalize(q, dim=-1), F.normalize(k, dim=-1)
        attn = (q @ k.transpose(-2, -1)) * self.temperature
        attn = attn.softmax(dim=-1)
        out = (attn @ v).reshape(B, C, H, W)
        return self.proj(out)

class GDFN(nn.Module):
    def __init__(self, dim, ffn_expansion=2.66):
        super().__init__()
        hidden = int(dim * ffn_expansion)
        self.net = nn.Sequential(
            nn.Conv2d(dim, hidden * 2, 1, bias=False),
            nn.Conv2d(hidden * 2, hidden * 2, 3, padding=1, groups=hidden * 2, bias=False),
        )
        self.proj = nn.Conv2d(hidden, dim, 1, bias=False)
    
    def forward(self, x):
        x1, x2 = self.net(x).chunk(2, dim=1)
        return self.proj(F.gelu(x1) * x2)

class SpectralTransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.norm1 = LayerNorm2d(dim)
        self.attn = MDTA(dim, num_heads)
        self.norm2 = LayerNorm2d(dim)
        self.ffn = GDFN(dim)
    
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

## 🏗️ PISTUNet

In [ ]:
class PISTUNet(nn.Module):
    def __init__(self, in_ch=58, out_ch=29, base=64, num_heads=4, num_blocks=[2, 2, 4, 2]):
        super().__init__()
        dims = [base, base*2, base*4, base*8]
        
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, dims[0], 3, 1, 1, bias=False),
            LayerNorm2d(dims[0]), nn.GELU(),
        )
        
        self.enc_blocks = nn.ModuleList()
        self.downsample = nn.ModuleList()
        for i, (dim, nb) in enumerate(zip(dims, num_blocks)):
            self.enc_blocks.append(nn.Sequential(
                *[SpectralTransformerBlock(dim, num_heads) for _ in range(nb)]
            ))
            if i < len(dims) - 1:
                self.downsample.append(nn.Sequential(
                    nn.Conv2d(dim, dims[i+1], 4, 2, 1, bias=False),
                    LayerNorm2d(dims[i+1]),
                ))
        
        self.bottleneck = nn.Sequential(
            *[SpectralTransformerBlock(dims[-1], num_heads) for _ in range(4)]
        )
        
        self.upsample = nn.ModuleList()
        self.fusion = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        for i in range(len(dims) - 1, 0, -1):
            self.upsample.append(nn.Sequential(
                nn.ConvTranspose2d(dims[i], dims[i-1], 2, 2, bias=False),
                LayerNorm2d(dims[i-1]),
            ))
            self.fusion.append(nn.Conv2d(dims[i-1] * 2, dims[i-1], 1, bias=False))
            self.dec_blocks.append(nn.Sequential(
                *[SpectralTransformerBlock(dims[i-1], num_heads) for _ in range(num_blocks[i-1])]
            ))
        
        self.spec_attn = SpectralChannelAttention(dims[0])
        self.spat_attn = SpatialAttention()
        self.head = nn.Sequential(
            nn.Conv2d(dims[0], dims[0], 3, 1, 1, bias=False), nn.GELU(),
            nn.Conv2d(dims[0], out_ch, 1),
        )
    
    def forward(self, x):
        init = x[:, :29]
        feat = self.stem(x)
        
        skips = []
        for i, enc in enumerate(self.enc_blocks):
            feat = enc(feat)
            if i < len(self.downsample):
                skips.append(feat)
                feat = self.downsample[i](feat)
        
        feat = self.bottleneck(feat)
        
        for up, fuse, dec, skip in zip(
            self.upsample, self.fusion, self.dec_blocks, reversed(skips)):
            feat = up(feat)
            if feat.shape != skip.shape:
                feat = F.interpolate(feat, skip.shape[2:], mode='bilinear', align_corners=False)
            feat = fuse(torch.cat([feat, skip], 1))
            feat = dec(feat)
        
        feat = self.spec_attn(feat)
        feat = self.spat_attn(feat)
        return self.head(feat) + init

# Param count
_m = PISTUNet(cfg.IN_CHANNELS, cfg.OUT_CHANNELS, cfg.BASE_DIM, cfg.NUM_HEADS, cfg.NUM_BLOCKS)
print(f"PISTUNet: {sum(p.numel() for p in _m.parameters() if p.requires_grad)/1e6:.2f}M params")
del _m

## 📉 Losses (NaN-safe)

**Key fix:** SAM loss uses `1 - cos_similarity` instead of `acos(cos_similarity)`. 
The `acos` function has infinite gradient at ±1, causing NaN in float16.

In [ ]:
class SSIMLoss(nn.Module):
    def __init__(self, window_size=7):
        super().__init__()
        self.ws = window_size
        self.C1, self.C2 = 0.01**2, 0.03**2
    
    def forward(self, pred, gt):
        pred, gt = pred.float(), gt.float()
        pad = self.ws // 2
        mu_p = F.avg_pool2d(pred, self.ws, 1, pad)
        mu_g = F.avg_pool2d(gt, self.ws, 1, pad)
        sig_p = torch.clamp(F.avg_pool2d(pred**2, self.ws, 1, pad) - mu_p**2, min=0)
        sig_g = torch.clamp(F.avg_pool2d(gt**2, self.ws, 1, pad) - mu_g**2, min=0)
        sig_pg = F.avg_pool2d(pred * gt, self.ws, 1, pad) - mu_p * mu_g
        ssim = ((2*mu_p*mu_g + self.C1) * (2*sig_pg + self.C2)) / \
               ((mu_p**2 + mu_g**2 + self.C1) * (sig_p + sig_g + self.C2) + 1e-8)
        return 1.0 - ssim.mean()


class SAMLoss(nn.Module):
    """Cosine distance loss — stable replacement for acos-based SAM."""
    def forward(self, pred, gt, eps=1e-6):
        pred, gt = pred.float(), gt.float()
        B, C, H, W = pred.shape
        p = pred.reshape(B, C, -1)
        g = gt.reshape(B, C, -1)
        dot = (p * g).sum(dim=1)
        p_norm = torch.sqrt((p**2).sum(dim=1) + eps)
        g_norm = torch.sqrt((g**2).sum(dim=1) + eps)
        cos_sim = torch.clamp(dot / (p_norm * g_norm + eps), 0.0, 1.0)
        return (1.0 - cos_sim).mean()


class ForwardConsistencyLoss(nn.Module):
    def __init__(self, mask_cube):
        super().__init__()
        self.register_buffer('mask', mask_cube)
    
    def forward(self, pred, coded):
        pred, coded = pred.float(), coded.float()
        mask = self.mask.unsqueeze(0).expand(pred.shape[0], -1, -1, -1)
        y_recon = (pred * mask).sum(dim=1, keepdim=True) / 29.0
        if coded.dim() == 3:
            coded = coded.unsqueeze(1)
        return F.l1_loss(y_recon, coded)


class CombinedLoss(nn.Module):
    def __init__(self, mask_cube, l1_w=1.0, ssim_w=0.3, sam_w=0.1, fwd_w=0.05):
        super().__init__()
        self.l1_w, self.ssim_w, self.sam_w, self.fwd_w = l1_w, ssim_w, sam_w, fwd_w
        self.ssim_loss = SSIMLoss()
        self.sam_loss = SAMLoss()
        self.fwd_loss = ForwardConsistencyLoss(mask_cube)
    
    def forward(self, pred, gt, coded):
        pred_f, gt_f = pred.float(), gt.float()
        loss_l1 = F.l1_loss(pred_f, gt_f)
        loss_ssim = self.ssim_loss(pred_f, gt_f)
        loss_sam = self.sam_loss(pred_f, gt_f)
        loss_fwd = self.fwd_loss(pred_f, coded)
        total = self.l1_w * loss_l1 + self.ssim_w * loss_ssim + self.sam_w * loss_sam + self.fwd_w * loss_fwd
        return total, {'l1': loss_l1.item(), 'ssim': loss_ssim.item(),
                       'sam': loss_sam.item(), 'fwd': loss_fwd.item(), 'total': total.item()}

## 📊 Eval Metrics (exact competition code)

In [ ]:
@torch.no_grad()
def calc_psnr(pred, gt, eps=1e-8):
    mse = F.mse_loss(pred.float(), gt.float())
    if mse < eps: return 50.0
    return (10.0 * torch.log10(1.0 / (mse + eps))).item()

@torch.no_grad()
def calc_ssim(pred, gt, window_size=7):
    pred, gt = pred.float(), gt.float()
    C1, C2 = 0.01**2, 0.03**2
    pad = window_size // 2
    mu_p = F.avg_pool2d(pred, window_size, 1, pad)
    mu_g = F.avg_pool2d(gt, window_size, 1, pad)
    sig_p = torch.clamp(F.avg_pool2d(pred**2, window_size, 1, pad) - mu_p**2, min=0)
    sig_g = torch.clamp(F.avg_pool2d(gt**2, window_size, 1, pad) - mu_g**2, min=0)
    sig_pg = F.avg_pool2d(pred * gt, window_size, 1, pad) - mu_p * mu_g
    ssim = ((2*mu_p*mu_g + C1) * (2*sig_pg + C2)) / \
           ((mu_p**2 + mu_g**2 + C1) * (sig_p + sig_g + C2) + 1e-8)
    return ssim.mean().item()

@torch.no_grad()
def calc_sam(pred, gt, eps=1e-8):
    pred, gt = pred.float(), gt.float()
    B, C, H, W = pred.shape
    p, g = pred.reshape(B, C, -1), gt.reshape(B, C, -1)
    dot = (p * g).sum(1)
    cos = torch.clamp(dot / (torch.norm(p, 2, 1) * torch.norm(g, 2, 1) + eps), -1+eps, 1-eps)
    return (torch.acos(cos) * 180.0 / math.pi).mean().item()

@torch.no_grad()
def aggregate_score(psnr, ssim, sam):
    return min(1.0, 0.5 * psnr / 50 + 0.25 * ssim + 0.25 * (1 - sam / 90))

In [ ]:
class CosineWarmupScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup = warmup_epochs
        self.total = total_epochs
        self.min_lr = min_lr
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.step(0)  # Apply warmup immediately
    
    def step(self, epoch):
        if epoch < self.warmup:
            factor = (epoch + 1) / self.warmup
        else:
            progress = (epoch - self.warmup) / max(1, self.total - self.warmup)
            factor = 0.5 * (1 + math.cos(math.pi * progress))
        for pg, blr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = max(self.min_lr, blr * factor)

## 🏋️ Training (NaN-safe)

In [ ]:
def train():
    print("=" * 60)
    print("CASSI GOAT 1.0 - SOTA Pipeline v2")
    print("=" * 60)
    
    cfg.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    cfg.PRED_DIR.mkdir(parents=True, exist_ok=True)
    
    full_ds = CASSIDataset(cfg.TRAIN_CODED_DIR, cfg.TRAIN_HS_DIR, mask_cube, augment=True)
    n_val = int(cfg.VAL_RATIO * len(full_ds))
    n_train = len(full_ds) - n_val
    train_set, val_set = random_split(full_ds, [n_train, n_val],
                                       generator=torch.Generator().manual_seed(cfg.SEED))
    
    val_ds_clean = CASSIDataset(cfg.TRAIN_CODED_DIR, cfg.TRAIN_HS_DIR, mask_cube, augment=False)
    val_set_clean = torch.utils.data.Subset(val_ds_clean, val_set.indices)
    
    train_dl = DataLoader(train_set, cfg.BATCH_SIZE, True, num_workers=cfg.NUM_WORKERS,
                           pin_memory=True, drop_last=True)
    val_dl = DataLoader(val_set_clean, cfg.BATCH_SIZE, False, num_workers=cfg.NUM_WORKERS,
                         pin_memory=True)
    print(f"Train: {n_train} | Val: {n_val}")
    
    # Data sanity check
    s_inp, s_tgt, s_cod = full_ds[0]
    print(f"Input: {s_inp.shape} [{s_inp.min():.4f}, {s_inp.max():.4f}]")
    print(f"Target: {s_tgt.shape} [{s_tgt.min():.4f}, {s_tgt.max():.4f}]")
    print(f"Coded: {s_cod.shape} [{s_cod.min():.4f}, {s_cod.max():.4f}]")
    
    model = PISTUNet(cfg.IN_CHANNELS, cfg.OUT_CHANNELS, cfg.BASE_DIM,
                     cfg.NUM_HEADS, cfg.NUM_BLOCKS).to(device)
    print(f"Params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = CosineWarmupScheduler(optimizer, cfg.WARMUP_EPOCHS, cfg.EPOCHS, cfg.MIN_LR)
    criterion = CombinedLoss(mask_cube.to(device), cfg.L1_WEIGHT, cfg.SSIM_WEIGHT,
                              cfg.SAM_WEIGHT, cfg.FORWARD_WEIGHT)
    scaler = torch.amp.GradScaler('cuda')
    
    best_score = 0
    nan_count = 0
    history = {'train_loss': [], 'val_psnr': [], 'val_ssim': [], 'val_sam': [], 'val_score': []}
    
    for epoch in range(cfg.EPOCHS):
        scheduler.step(epoch)
        cur_lr = optimizer.param_groups[0]['lr']
        
        model.train()
        epoch_loss, n_batches = 0, 0
        optimizer.zero_grad()
        
        pbar = tqdm(train_dl, desc=f"Epoch {epoch+1}/{cfg.EPOCHS}")
        for step, (inp, tgt, coded) in enumerate(pbar):
            inp = inp.to(device, non_blocking=True)
            tgt = tgt.to(device, non_blocking=True)
            coded = coded.to(device, non_blocking=True)
            
            with torch.amp.autocast('cuda'):
                pred = torch.clamp(model(inp), 0, 1)
                loss, ld = criterion(pred, tgt, coded)
                loss = loss / cfg.GRAD_ACCUM
            
            if torch.isnan(loss) or torch.isinf(loss):
                nan_count += 1
                optimizer.zero_grad()
                if nan_count > 50:
                    print(f"\n⚠️ Too many NaN batches ({nan_count}), stopping!")
                    return model, history
                continue
            
            scaler.scale(loss).backward()
            
            if (step + 1) % cfg.GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                if torch.isnan(grad_norm) or torch.isinf(grad_norm):
                    optimizer.zero_grad()
                    scaler.update()  # MUST update scaler to reset its state
                    nan_count += 1
                    continue
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            epoch_loss += ld['total']
            n_batches += 1
            pbar.set_postfix(loss=f"{ld['total']:.4f}", lr=f"{cur_lr:.1e}")
        
        history['train_loss'].append(epoch_loss / max(n_batches, 1))
        
        # Validate
        model.eval()
        vp, vs, va, nb = 0, 0, 0, 0
        with torch.no_grad():
            for inp, tgt, coded in val_dl:
                inp, tgt = inp.to(device), tgt.to(device)
                with torch.amp.autocast('cuda'):
                    pred = torch.clamp(model(inp).float(), 0, 1)
                vp += calc_psnr(pred, tgt)
                vs += calc_ssim(pred, tgt)
                va += calc_sam(pred, tgt)
                nb += 1
        
        vp, vs, va = vp/nb, vs/nb, va/nb
        score = aggregate_score(vp, vs, va)
        history['val_psnr'].append(vp)
        history['val_ssim'].append(vs)
        history['val_sam'].append(va)
        history['val_score'].append(score)
        
        print(f"  → PSNR={vp:.2f}dB | SSIM={vs:.4f} | SAM={va:.2f}° | Score={score:.4f} | NaNs={nan_count}")
        
        if score > best_score and not math.isnan(vp):
            best_score = score
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'score': score, 'psnr': vp, 'ssim': vs, 'sam': va},
                       cfg.CHECKPOINT_DIR / "best.pth")
            print(f"  ★ New best! Score={score:.4f}")
        
        torch.save(model.state_dict(), cfg.CHECKPOINT_DIR / "latest.pth")
    
    print(f"\nDone! Best score: {best_score:.4f}")
    return model, history

model, history = train()

## 📈 Training Curves

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].plot(history['train_loss']); axes[0].set_title('Train Loss')
axes[1].plot(history['val_psnr']); axes[1].set_title('Val PSNR (dB)')
axes[2].plot(history['val_ssim']); axes[2].set_title('Val SSIM')
axes[3].plot(history['val_score']); axes[3].set_title('Val Score')
for ax in axes: ax.grid(True, alpha=0.3); ax.set_xlabel('Epoch')
plt.tight_layout(); plt.savefig(cfg.OUTPUT_DIR / "training_curves.png", dpi=150); plt.show()

## 🔮 Inference + TTA

In [ ]:
@torch.no_grad()
def tta_predict(model, x):
    preds = []
    for flip in [False, True]:
        for k in range(4):
            x_aug = x.clone()
            if flip: x_aug = torch.flip(x_aug, [-1])
            if k > 0: x_aug = torch.rot90(x_aug, k, [-2, -1])
            with torch.amp.autocast('cuda'):
                pred = model(x_aug).float()
            if k > 0: pred = torch.rot90(pred, -k, [-2, -1])
            if flip: pred = torch.flip(pred, [-1])
            preds.append(pred)
    return torch.stack(preds).mean(0)

@torch.no_grad()
def inference(model):
    cfg.PRED_DIR.mkdir(parents=True, exist_ok=True)
    ckpt = cfg.CHECKPOINT_DIR / "best.pth"
    if ckpt.exists():
        sd = torch.load(ckpt, weights_only=False)
        model.load_state_dict(sd['model_state_dict'])
        print(f"Loaded best (Score={sd['score']:.4f}, PSNR={sd['psnr']:.2f}dB)")
    
    model.eval()
    test_ds = CASSITestDataset(cfg.TEST_CODED_DIR, mask_cube)
    test_dl = DataLoader(test_ds, 1, False, num_workers=cfg.NUM_WORKERS)
    
    for inp, coded, stem in tqdm(test_dl, desc="Inference"):
        inp = inp.to(device)
        pred = tta_predict(model, inp) if cfg.USE_TTA else model(inp).float()
        pred = torch.clamp(pred.squeeze(0).cpu(), 0, 1)
        idx = int(stem[0].split("_")[-1])
        torch.save(pred, cfg.PRED_DIR / f"sample_{idx:04d}.pt")
    
    n = len(list(cfg.PRED_DIR.glob("*.pt")))
    s = torch.load(cfg.PRED_DIR / "sample_0000.pt", weights_only=True)
    print(f"Saved {n} files | Shape: {s.shape} | Range: [{s.min():.4f}, {s.max():.4f}]")
    assert s.shape == (29, 96, 96) and n == 300
    print("✓ Submission ready!")

inference(model)

## 📤 Submit

In [ ]:
# !git clone --depth 1 https://github.com/chater-marzougui/MLS-GOAT.git
# !mv MLS-GOAT/GOAT .
# !rm -rf MLS-GOAT

# import sys; sys.path.insert(0, '/kaggle/working')
# from GOAT import submit, leaderboard
# submit.challenge1(str(cfg.PRED_DIR), "YOUR_TEAM", "YOUR_PASSWORD")
# leaderboard.getLB_chall1("YOUR_TEAM")